# CEPHA29-MD — masked multi-domain fine-tune (all 29 landmarks)

Fine-tunes the 29-landmark model on **Aariz (all 29)** + **ISBI 2015 (only the 19
shared)**, using a per-sample visibility mask so the 10 Aariz-only landmarks are
not supervised on ISBI images. This keeps full 29-landmark production parity while
gaining the domain diversity that made E1 (the 19-landmark model) a clear win.

- Vocabulary: CEPHA29 (unchanged channel order).
- Dataset: `Merged29` annotations; ISBI JSONs carry `visible: [19 symbols]`.
- Patches: `scripts/ai/cephalohrnet_mask_patch.py` (dataset mask, masked loss,
  masked metrics).
- Starts from the original 29-landmark `best.pt` (head loaded too — shapes match).

**Kaggle setup**: GPU T4 x2, Internet On, attach `best.pt` (or `best.zip`).
ISBI/Aariz are downloaded automatically.

Outputs: `/kaggle/working/deliverables_md29/`.

In [ ]:
!pip -q install timm albumentations opencv-python-headless pyyaml onnx onnxruntime pillow numpy scipy pandas requests seaborn
print("deps installed")

In [ ]:
import os
%cd /kaggle/working
!rm -rf CephaloHRNet CephalometryStudio
!git clone --depth 1 https://github.com/Cestovatels/CephaloHRNet.git
!git clone --depth 1 https://github.com/Muhammad-Nabeel-Sh/CephalometryStudio.git
assert os.path.exists("CephaloHRNet/models/hrnet.py")
assert os.path.exists("CephalometryStudio/scripts/ai/cephalohrnet_mask_patch.py"), "push the mask patch first"
print("cloned")

In [ ]:
import glob, os, sys, urllib.request
import numpy as np
sys.path.insert(0, "CephalometryStudio/scripts/ai")
from canonical import CEPHA29, ISBI19, ISBI_TO_CEPHA

BASE = "https://raw.githubusercontent.com/stolariks/medical-landmark-detection/main/data/isbi-2015/"
ANN_TRAIN = "/kaggle/working/isbi_train.csv"
ANN_TEST = "/kaggle/working/isbi_test.csv"
for url, dest in [(BASE + "train/annotations.csv", ANN_TRAIN),
                  (BASE + "test/annotations.csv", ANN_TEST)]:
    if not os.path.exists(dest):
        req = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
        with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
            f.write(r.read())

def read_isbi_csv(path):
    import csv
    out = {}
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.reader(f):
            if not row or row[0] == "image_path":
                continue
            try:
                pts = np.array([float(v) for v in row[1:]], np.float64).reshape(-1, 2)
            except Exception:
                continue
            if pts.shape[0] == len(ISBI19):
                out[os.path.splitext(os.path.basename(row[0]))[0]] = pts
    return out

isbi_gt = {"train": read_isbi_csv(ANN_TRAIN), "test": read_isbi_csv(ANN_TEST)}
print("ISBI GT:", {k: len(v) for k, v in isbi_gt.items()})

def find_input_images(stems):
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for p in glob.glob("/kaggle/input/**/" + ext, recursive=True):
            if os.path.splitext(os.path.basename(p))[0] in stems:
                return os.path.dirname(p)
    return None

ISBI_DIR = find_input_images(set(isbi_gt["train"]) | set(isbi_gt["test"]))
print("ISBI image dir:", ISBI_DIR or "(fetch from mirror)")

def isbi_image(split, stem):
    if ISBI_DIR:
        for ext in (".jpg", ".jpeg", ".png"):
            p = os.path.join(ISBI_DIR, stem + ext)
            if os.path.exists(p):
                return p
        return None
    cache = "/kaggle/working/isbi_img_" + split
    os.makedirs(cache, exist_ok=True)
    for ext in (".jpg", ".png"):
        dest = os.path.join(cache, stem + ext)
        if os.path.exists(dest):
            return dest
        try:
            url = BASE + split + "/cepha400/" + stem + ext
            req = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
                f.write(r.read())
            return dest
        except Exception:
            continue
    return None

isbi_cases = {"train": [], "test": []}
for split in ("train", "test"):
    for stem, gt in isbi_gt[split].items():
        p = isbi_image(split, stem)
        if p:
            isbi_cases[split].append((stem, p, gt))
    print("ISBI", split, "cases:", len(isbi_cases[split]))

def find_dataset(root):
    for base, dirs, _ in os.walk(root):
        for split in ("train", "valid"):
            if os.path.isdir(os.path.join(base, split, "Cephalograms")):
                return base
    return None

AARIZ = find_dataset("/kaggle/input")
if AARIZ is None:
    import zipfile
    URL = "https://ndownloader.figshare.com/files/51041642"
    ZIP = "/kaggle/working/Aariz.zip"
    if not os.path.exists(ZIP) or os.path.getsize(ZIP) != 2098209792:
        req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=900) as r, open(ZIP, "wb") as f:
            while True:
                c = r.read(1 << 20)
                if not c:
                    break
                f.write(c)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/kaggle/working/aariz")
    AARIZ = find_dataset("/kaggle/working/aariz")
print("AARIZ:", AARIZ)
assert AARIZ, "Aariz dataset not found"

In [ ]:
import glob, json, os, shutil
import numpy as np

MD = "/kaggle/working/md29"
EVAL = "/kaggle/working/eval29"
for d in (MD, EVAL):
    shutil.rmtree(d, ignore_errors=True)

def ann_dir(root):
    # CephaloHRNet resolves --annotators Merged29 under this directory.
    return os.path.join(root, "Annotations", "Cephalometric Landmarks", "Merged29")

def read_syms(path):
    data = json.load(open(path, encoding="utf-8"))
    return {lm["symbol"]: (float(lm["value"]["x"]), float(lm["value"]["y"]))
            for lm in data.get("landmarks", []) if "x" in (lm.get("value") or {})}

def aariz_avg(split, stem):
    root = os.path.join(AARIZ, split, "Annotations", "Cephalometric Landmarks")
    dirs = [d for d in glob.glob(os.path.join(root, "*")) if os.path.isdir(d)]
    js = [os.path.join(d, stem + ".json") for d in dirs
          if os.path.exists(os.path.join(d, stem + ".json"))]
    if not js:
        return None
    sets = [read_syms(p) for p in js]
    keys = set(sets[0])
    for s in sets[1:]:
        keys &= set(s)
    return {k: (float(np.mean([s[k][0] for s in sets])),
                float(np.mean([s[k][1] for s in sets]))) for k in keys}

def write_case(root, prefix, domain, stem, img_path, pts29, visible=None):
    ceph = os.path.join(root, "Cephalograms")
    ad = ann_dir(root)
    os.makedirs(ceph, exist_ok=True)
    os.makedirs(ad, exist_ok=True)
    new_stem = (domain + "_" + stem) if prefix else stem
    ext = os.path.splitext(img_path)[1].lower()
    if ext not in (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"):
        ext = ".jpg"
    dst = os.path.join(ceph, new_stem + ext)
    if not os.path.exists(dst):
        shutil.copy(img_path, dst)
    doc = {"ceph_id": new_stem,
           "landmarks": [{"symbol": s, "value": {"x": pts29[s][0], "y": pts29[s][1]}} for s in CEPHA29]}
    if visible is not None:
        doc["visible"] = visible
    json.dump(doc, open(os.path.join(ad, new_stem + ".json"), "w"))

def aariz_pts(split, stem):
    avg = aariz_avg(split, stem)
    if not avg or not all(s in avg for s in CEPHA29):
        return None
    return {s: avg[s] for s in CEPHA29}

def isbi_pts(gt):
    # Map the 19 ISBI landmarks onto CEPHA29; the other 10 are masked sentinels.
    mapped = {ISBI_TO_CEPHA[s]: (float(gt[i][0]), float(gt[i][1])) for i, s in enumerate(ISBI19)}
    pts = {s: mapped.get(s, (-1.0, -1.0)) for s in CEPHA29}
    visible = [s for s in CEPHA29 if s in mapped]
    return pts, visible

for split in ("train", "valid"):
    for p in sorted(glob.glob(os.path.join(AARIZ, split, "Cephalograms", "*"))):
        stem = os.path.splitext(os.path.basename(p))[0]
        pts = aariz_pts(split, stem)
        if pts:
            write_case(os.path.join(MD, split), True, "aariz", stem, p, pts)
    src_split = "train" if split == "train" else "test"
    for stem, p, gt in isbi_cases[src_split]:
        pts, visible = isbi_pts(gt)
        write_case(os.path.join(MD, split), True, "isbi", stem, p, pts, visible)

for p in sorted(glob.glob(os.path.join(AARIZ, "valid", "Cephalograms", "*"))):
    stem = os.path.splitext(os.path.basename(p))[0]
    pts = aariz_pts("valid", stem)
    if pts:
        write_case(os.path.join(EVAL, "aariz29"), False, "aariz", stem, p, pts)
for stem, p, gt in isbi_cases["test"]:
    pts, visible = isbi_pts(gt)
    write_case(os.path.join(EVAL, "isbi19"), False, "isbi", stem, p, pts, visible)
if os.path.isdir(os.path.join(EVAL, "isbi19")):
    shutil.copy(ANN_TEST, os.path.join(EVAL, "isbi19", "annotations.csv"))

def count(root):
    return len(glob.glob(os.path.join(root, "Cephalograms", "*")))
print("md29 train:", count(os.path.join(MD, "train")),
      "valid:", count(os.path.join(MD, "valid")),
      "| eval aariz29:", count(os.path.join(EVAL, "aariz29")),
      "isbi19:", count(os.path.join(EVAL, "isbi19")))
assert count(os.path.join(MD, "train")) > 0 and count(os.path.join(MD, "valid")) > 0

In [ ]:
import sys
sys.path.insert(0, "CephalometryStudio/scripts/ai")
from cephalohrnet_mask_patch import apply_mask_patches
apply_mask_patches("/kaggle/working/CephaloHRNet")
print("mask patches applied")

In [ ]:
import glob, os, shutil
import torch

pref = {"best.pt": 0, "best.pth": 1, "best.zip": 2}
cands = []
for name, rank in pref.items():
    for p in glob.glob("/kaggle/input/**/" + name, recursive=True):
        cands.append((rank, p))
cands.sort()
BEST = cands[0][1] if cands else None
if BEST and BEST.lower().endswith(".zip"):
    dest = "/kaggle/working/best.pt"
    if not os.path.exists(dest):
        shutil.copy(BEST, dest)
    BEST = dest
print("best.pt:", BEST)
assert BEST, "Attach the 29-landmark best.pt (or best.zip) as a Kaggle dataset"
ck = torch.load(BEST, map_location="cpu", weights_only=False)
n_head = ck["model"]["head.weight"].shape[0]
print("checkpoint head channels:", n_head, "| epoch:", ck.get("epoch"))
assert n_head == 29, "expected the 29-landmark checkpoint"

In [ ]:
import glob, os, shutil
calib = "/kaggle/working/calib_md29"
os.makedirs(calib, exist_ok=True)
imgs = sorted(glob.glob(os.path.join(MD, "train", "Cephalograms", "*")))[:16]
for f in imgs:
    shutil.copy(f, os.path.join(calib, os.path.basename(f)))
print("calib:", len(imgs))

In [ ]:
import glob, os, subprocess, sys

for split in ("train", "valid"):
    n_img = len(glob.glob(os.path.join(MD, split, "Cephalograms", "*")))
    n_ann = len(glob.glob(os.path.join(MD, split, "Annotations", "Cephalometric Landmarks", "Merged29", "*.json")))
    print(f"{split}: images={n_img} annotations={n_ann}")
    assert n_img and n_ann, f"{split} split is empty"

cmd = [sys.executable, "train.py",
       "--data", MD, "--model", "hrnet_w32",
       "--imgsz", "768", "--batch", "4", "--epochs", "80",
       "--lr", "5e-4",
       "--loss", "awing", "--amp", "--device", "0",
       "--workers", "0", "--patience", "15",
       "--project", "runs/train", "--name", "md29_w32",
       "--annotators", "Merged29", "--img-dir", "Cephalograms",
       "--pretrained-path", BEST]
print(" ".join(cmd))
r = subprocess.run(cmd, cwd="/kaggle/working/CephaloHRNet", capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-6000:])
r.check_returncode()

In [ ]:
import glob, os, subprocess, sys
BESTR = sorted(glob.glob("/kaggle/working/CephaloHRNet/runs/train/md29_*/best.pt"),
               key=os.path.getmtime)[-1]
SAMPLE = sorted(glob.glob(os.path.join(MD, "train", "Cephalograms", "*")))[0]
print("weights:", BESTR)
subprocess.run([sys.executable, "CephalometryStudio/scripts/export-landmark-onnx.py",
                "--repo", "/kaggle/working/CephaloHRNet",
                "--weights", BESTR, "--model", "hrnet_w32",
                "--num-landmarks", "29", "--input-size", "768",
                "--out", "/kaggle/working/landmarks-md29.onnx",
                "--quantize", "--calib-dir", "/kaggle/working/calib_md29",
                "--verify", SAMPLE],
               cwd="/kaggle/working", check=True)

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "CephalometryStudio/scripts/build-shape-model.py",
                "--set", "cepha29",
                "--ann-dir", os.path.join(AARIZ, "train", "Annotations", "Cephalometric Landmarks", "Senior Orthodontists"),
                "--k", "10",
                "--out", "/kaggle/working/shapeModel.cepha29.json"],
               cwd="/kaggle/working", check=True)

In [ ]:
import os, subprocess, sys
OUT = "/kaggle/working/audit_md29"
os.makedirs(OUT, exist_ok=True)
ERR = []

# External: ISBI test (19 shared), raw, production search.
r = subprocess.run([sys.executable, "CephalometryStudio/scripts/ai/external_benchmark.py",
                    "--model", "/kaggle/working/landmarks-md29.int8.onnx",
                    "--isbi-images", os.path.join(EVAL, "isbi19", "Cephalograms"),
                    "--isbi-ann", os.path.join(EVAL, "isbi19", "annotations.csv"),
                    "--search", "--no-ssm",
                    "--out", os.path.join(OUT, "isbi19")],
                   cwd="/kaggle/working")
if r.returncode: ERR.append("isbi")

# In-domain: Aariz valid (29), raw.
r = subprocess.run([sys.executable, "CephalometryStudio/scripts/eval-landmark-model.py",
                    "--model", "/kaggle/working/landmarks-md29.int8.onnx",
                    "--set", "cepha29", "--limit", "200", "--dark",
                    "--ann-dir", os.path.join(EVAL, "aariz29", "Annotations", "Cephalometric Landmarks", "Merged29"),
                    "--images", os.path.join(EVAL, "aariz29", "Cephalograms"),
                    "--metrics-out", os.path.join(OUT, "aariz29_metrics.json")],
                   cwd="/kaggle/working")
if r.returncode: ERR.append("aariz")
print("failed:", ERR)

In [ ]:
import glob, hashlib, os, shutil
out = "/kaggle/working/deliverables_md29"
os.makedirs(out, exist_ok=True)
for name in ("landmarks-md29.onnx", "landmarks-md29.int8.onnx", "landmarks-md29-verify.png",
             "shapeModel.cepha29.json"):
    for p in glob.glob("/kaggle/working/" + name):
        shutil.copy(p, out)
for p in glob.glob("/kaggle/working/audit_md29/**/*", recursive=True):
    if os.path.isfile(p):
        rel = os.path.relpath(p, "/kaggle/working/audit_md29")
        dst = os.path.join(out, "audit", rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(p, dst)

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

print("=== DELIVERABLES ===")
for f in sorted(os.listdir(out)):
    p = os.path.join(out, f)
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:9.2f} MB  {f}\n            sha256 {sha(p)}")

## Next

1. Download `/kaggle/working/deliverables_md29/`.
2. Compare `audit/isbi19/isbi_metrics.json` with the E1 19-model (SDR@2 64.7%,
   median 1.34) and the 29-model baseline (50.6%, 1.97).
3. Compare `audit/aariz29_metrics.json` with the original 29-model baseline
   (Aariz valid: MRE ~2.55 mm with SSM; this run is raw).
4. If it matches or beats E1 on ISBI while keeping all 29 landmarks, it becomes the
   production model (switch `LANDMARK_MODEL` back to a 29-set config).
5. Inspect `landmarks-md29-verify.png` (29 channels, CEPHA29 order).